# Territorial Risk Intelligence

## Data Preparation Workflow

This notebook demonstrates the data preparation process applied to household survey data, including:

- Data ingestion
- Header standardization
- Data cleaning
- Identifier generation
- Feature engineering
- Sociodemographic enrichment

In [ ]:
# =============================================================================
# IMPORT LIBRARIES
# =============================================================================

import pandas as pd
import numpy as np

## Load Raw Survey Data

The original survey data was collected through structured household interviews and stored in Excel format.

In [ ]:
# =============================================================================
# INGEST RAW EXCEL FILE
# =============================================================================

df = pd.read_excel("../data/raw/sample_survey_raw.xlsx")

df.head()

## Standardize Headers

Column names are converted to lowercase and spaces are replaced with underscores to ensure consistency throughout the pipeline.

In [ ]:
# =============================================================================
# STANDARDIZE COLUMN NAMES
# =============================================================================

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

df.columns

## Trim Text Fields

Leading and trailing spaces are removed from all text columns.

In [ ]:
# =============================================================================
# TRIM TEXT COLUMNS
# =============================================================================

text_columns = df.select_dtypes(include="object").columns

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

## Create Household Identifier

A unique household ID is generated using the watershed prefix and the original survey number.

Example:

- River + 65 → RI65
- Stream + 12 → ST12

In [ ]:
# =============================================================================
# CREATE HOUSEHOLD ID
# =============================================================================

df["id_household"] = (
    df["watershed"]
      .str.replace("Watershed_", "", regex=False)
      .str.zfill(2)
      .radd("WS")
      +
      df["survey_number"].astype(str).str.zfill(3)
)

df[["survey_number", "watershed", "id_household"]].head()

In [ ]:
# =============================================================================
# REMOVE ORIGINAL SURVEY NUMBER
# =============================================================================

df = df.drop(columns=["survey_number"])

## Create Age Groups

Residents are grouped into age brackets to support demographic analysis.

In [ ]:
# =============================================================================
# AGE GROUP
# =============================================================================

def age_group(age):

    if age < 20:
        return "< 20"
    elif age <= 29:
        return "20 to 29"
    elif age <= 39:
        return "30 to 39"
    elif age <= 49:
        return "40 to 49"
    elif age <= 59:
        return "50 to 59"
    else:
        return "60+"

df["age_group"] = df["head_age"].apply(age_group)

## Create Household Income Categories

Family income is categorized using minimum wage ranges.

In [ ]:
# =============================================================================
# FAMILY INCOME RANGE
# =============================================================================

MINIMUM_WAGE = 1518

def income_range(value):

    sm = value / MINIMUM_WAGE

    if sm <= 1:
        return "Up to 1 MW"
    elif sm <= 2:
        return "1 to 2 MW"
    elif sm <= 4:
        return "2 to 4 MW"
    elif sm <= 8:
        return "4 to 8 MW"
    else:
        return "> 8 MW"

df["family_income_range"] = (
    df["family_income"]
      .apply(income_range)
)

## Calculate Per Capita Income

Per capita income is calculated using household income and household size.

In [ ]:
# =============================================================================
# PER CAPITA INCOME
# =============================================================================

df["income_per_capita"] = (
    df["family_income"]
    /
    df["household_size"]
)

## Result

The resulting dataset is standardized, enriched, and ready for relational modeling and territorial risk analysis.

In [ ]:
# =============================================================================
# FINAL DATASET
# =============================================================================

df.head()